In [1]:
# ============================================================
# LEVIR-CC + LEVIR-CD
# SIAMESE U-NET + CHANGE CLASSIFICATION + CAPTIONING
# ============================================================

import os
import json
import ast
import random
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm


# ============================================================
# 1. CONFIG
# ============================================================

CSV_PATH = r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET\final_dataset.csv"

SEED = 42
BATCH_SIZE = 4
NUM_WORKERS = 0

IMAGE_SIZE = 256
MAX_LEN = 40

EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-5

SEG_WEIGHT = 1.0
CLS_WEIGHT = 0.5
CAPTION_WEIGHT = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


# ============================================================
# 2. SEED
# ============================================================

random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. LOAD DATASET
# ============================================================

df = pd.read_csv(CSV_PATH)

print("\nDataset:")
print("Total:", len(df))
print(df["split"].value_counts())

print("\nChange distribution:")
print(
    df.groupby(["split", "changeflag"]).size()
)


# ============================================================
# 4. PARSE CAPTIONS
# ============================================================

def parse_captions(x):

    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    try:
        return json.loads(x)
    except:
        try:
            return ast.literal_eval(x)
        except:
            return [str(x)]


df["captions"] = df["captions"].apply(parse_captions)


# ============================================================
# 5. BUILD VOCABULARY
# ============================================================

SPECIAL_TOKENS = [
    "<pad>",
    "<start>",
    "<end>",
    "<unk>"
]

counter = Counter()

for captions in df["captions"]:

    for caption in captions:

        tokens = caption.lower().strip().split()

        counter.update(tokens)


word2idx = {
    token: idx
    for idx, token in enumerate(SPECIAL_TOKENS)
}

for word, count in counter.items():

    if word not in word2idx:
        word2idx[word] = len(word2idx)


idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

PAD_IDX = word2idx["<pad>"]
START_IDX = word2idx["<start>"]
END_IDX = word2idx["<end>"]
UNK_IDX = word2idx["<unk>"]

VOCAB_SIZE = len(word2idx)

print("\nVocabulary size:", VOCAB_SIZE)


# ============================================================
# 6. DATASET
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
])


class LEVIRDataset(Dataset):

    def __init__(
        self,
        dataframe,
        max_len=40
    ):

        self.df = dataframe.reset_index(drop=True)
        self.max_len = max_len


    def encode_caption(self, captions):

        caption = random.choice(captions)

        tokens = caption.lower().strip().split()

        tokens = (
            [START_IDX]
            +
            [
                word2idx.get(
                    token,
                    UNK_IDX
                )
                for token in tokens
            ]
            +
            [END_IDX]
        )

        if len(tokens) > self.max_len:
            tokens = tokens[:self.max_len]

            tokens[-1] = END_IDX

        else:

            tokens += [
                PAD_IDX
            ] * (
                self.max_len - len(tokens)
            )

        return torch.tensor(
            tokens,
            dtype=torch.long
        )


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ----------------------------------------------------
        # Images
        # ----------------------------------------------------

        image_a = Image.open(
            row["cc_A"]
        ).convert("RGB")

        image_b = Image.open(
            row["cc_B"]
        ).convert("RGB")

        image_a = image_transform(
            image_a
        )

        image_b = image_transform(
            image_b
        )

        # ----------------------------------------------------
        # Mask
        # ----------------------------------------------------

        mask = Image.open(
            row["mask"]
        ).convert("L")

        mask = mask.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.NEAREST
        )

        mask = np.asarray(
            mask,
            dtype=np.float32
        )

        mask = (
            mask > 127
        ).astype(np.float32)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        # ----------------------------------------------------
        # Caption
        # ----------------------------------------------------

        caption = self.encode_caption(
            row["captions"]
        )

        # ----------------------------------------------------
        # Change flag
        # ----------------------------------------------------

        changeflag = torch.tensor(
            float(row["changeflag"]),
            dtype=torch.float32
        )

        return {
            "image_a": image_a,
            "image_b": image_b,
            "mask": mask,
            "caption": caption,
            "changeflag": changeflag
        }


# ============================================================
# 7. TRAIN / VALIDATION SPLIT
# ============================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "val"
].copy()

train_dataset = LEVIRDataset(
    train_df,
    MAX_LEN
)

val_dataset = LEVIRDataset(
    val_df,
    MAX_LEN
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


print("\nTrain:", len(train_dataset))
print("Val:", len(val_dataset))

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)


# ============================================================
# 8. VERIFY BATCH
# ============================================================

batch = next(iter(train_loader))

print("\n" + "=" * 60)
print("BATCH VERIFICATION")
print("=" * 60)

print(
    "Image A:",
    batch["image_a"].shape
)

print(
    "Image B:",
    batch["image_b"].shape
)

print(
    "Mask:",
    batch["mask"].shape
)

print(
    "Caption:",
    batch["caption"].shape
)

print(
    "Change flag:",
    batch["changeflag"].shape
)


# ============================================================
# 9. DOUBLE CONVOLUTION
# ============================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )


    def forward(self, x):

        return self.block(x)


# ============================================================
# 10. SIAMESE U-NET
# ============================================================

class SiameseUNet(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------------------------------
        # Shared encoder
        # ----------------------------------------------------

        self.enc1 = DoubleConv(
            3,
            32
        )

        self.enc2 = DoubleConv(
            32,
            64
        )

        self.enc3 = DoubleConv(
            64,
            128
        )

        self.enc4 = DoubleConv(
            128,
            256
        )

        self.pool = nn.MaxPool2d(
            2
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        # ----------------------------------------------------
        # Final output
        # ----------------------------------------------------

        self.mask_head = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

        # ----------------------------------------------------
        # Classification head
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d(
                1
            ),

            nn.Flatten(),

            nn.Linear(
                256,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                0.3
            ),

            nn.Linear(
                64,
                1
            )
        )


    def encode(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        return e1, e2, e3, e4


    def forward(self, image_a, image_b):

        # ----------------------------------------------------
        # Encode both images using SAME encoder
        # ----------------------------------------------------

        a1, a2, a3, a4 = self.encode(
            image_a
        )

        b1, b2, b3, b4 = self.encode(
            image_b
        )

        # ----------------------------------------------------
        # Difference features
        # ----------------------------------------------------

        d1 = torch.abs(
            a1 - b1
        )

        d2 = torch.abs(
            a2 - b2
        )

        d3 = torch.abs(
            a3 - b3
        )

        d4 = torch.abs(
            a4 - b4
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        x = self.up3(d4)

        x = torch.cat(
            [x, d3],
            dim=1
        )

        x = self.dec3(x)

        x = self.up2(x)

        x = torch.cat(
            [x, d2],
            dim=1
        )

        x = self.dec2(x)

        x = self.up1(x)

        x = torch.cat(
            [x, d1],
            dim=1
        )

        x = self.dec1(x)

        # ----------------------------------------------------
        # Segmentation
        # ----------------------------------------------------

        mask = self.mask_head(
            x
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        classification = self.classifier(
            d4
        )

        return (
            mask,
            classification,
            d4
        )


# ============================================================
# 11. CAPTION DECODER
# ============================================================

class CaptionDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        feature_dim=256,
        embed_dim=128,
        hidden_dim=256
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=PAD_IDX
        )

        self.init_h = nn.Linear(
            feature_dim,
            hidden_dim
        )

        self.init_c = nn.Linear(
            feature_dim,
            hidden_dim
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )


    def forward(
        self,
        features,
        captions
    ):

        # ----------------------------------------------------
        # Remove spatial dimensions
        # ----------------------------------------------------

        pooled = F.adaptive_avg_pool2d(
            features,
            1
        ).flatten(1)

        h = self.init_h(
            pooled
        ).unsqueeze(0)

        c = self.init_c(
            pooled
        ).unsqueeze(0)

        # Remove final <end> token
        inputs = captions[:, :-1]

        embeddings = self.embedding(
            inputs
        )

        outputs, _ = self.lstm(
            embeddings,
            (h, c)
        )

        logits = self.fc(
            outputs
        )

        return logits


# ============================================================
# 12. COMPLETE MODEL
# ============================================================

class ChangeCaptionModel(nn.Module):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.siamese = SiameseUNet()

        self.caption_decoder = CaptionDecoder(
            vocab_size
        )


    def forward(
        self,
        image_a,
        image_b,
        captions
    ):

        mask, classification, features = (
            self.siamese(
                image_a,
                image_b
            )
        )

        caption_logits = (
            self.caption_decoder(
                features,
                captions
            )
        )

        return (
            mask,
            classification,
            caption_logits
        )


# ============================================================
# 13. CREATE MODEL
# ============================================================

model = ChangeCaptionModel(
    VOCAB_SIZE
).to(DEVICE)


# ============================================================
# 14. PARAMETER COUNT
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "\nTrainable parameters:",
    f"{trainable_params / 1e6:.2f} M"
)


# ============================================================
# 15. MODEL SHAPE TEST
# ============================================================

model.eval()

with torch.no_grad():

    test_a = batch["image_a"][:2].to(
        DEVICE
    )

    test_b = batch["image_b"][:2].to(
        DEVICE
    )

    test_caption = batch["caption"][:2].to(
        DEVICE
    )

    pred_mask, pred_flag, pred_caption = (
        model(
            test_a,
            test_b,
            test_caption
        )
    )


print("\n" + "=" * 60)
print("MODEL VERIFICATION")
print("=" * 60)

print(
    "Mask:",
    pred_mask.shape
)

print(
    "Classification:",
    pred_flag.shape
)

print(
    "Caption:",
    pred_caption.shape
)

assert pred_mask.shape == (
    2,
    1,
    256,
    256
)

assert pred_flag.shape == (
    2,
    1
)

assert pred_caption.shape == (
    2,
    MAX_LEN - 1,
    VOCAB_SIZE
)

print("\nMODEL VERIFICATION: PASS")


# ============================================================
# 16. LOSSES
# ============================================================

bce_seg = nn.BCEWithLogitsLoss()

bce_cls = nn.BCEWithLogitsLoss()

caption_loss_fn = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)


def dice_loss(
    logits,
    targets,
    smooth=1.0
):

    probs = torch.sigmoid(
        logits
    )

    probs = probs.flatten(1)

    targets = targets.flatten(1)

    intersection = (
        probs * targets
    ).sum(dim=1)

    dice = (
        2 * intersection + smooth
    ) / (
        probs.sum(dim=1)
        +
        targets.sum(dim=1)
        +
        smooth
    )

    return (
        1 - dice
    ).mean()


def segmentation_loss(
    logits,
    targets
):

    return (
        0.5 * bce_seg(
            logits,
            targets
        )
        +
        0.5 * dice_loss(
            logits,
            targets
        )
    )


# ============================================================
# 17. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=4
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)


# ============================================================
# 18. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader
):

    model.train()

    total_loss = 0.0
    total_seg = 0.0
    total_cls = 0.0
    total_cap = 0.0

    progress = tqdm(
        loader,
        desc="Training"
    )

    for batch in progress:

        image_a = batch[
            "image_a"
        ].to(
            DEVICE,
            non_blocking=True
        )

        image_b = batch[
            "image_b"
        ].to(
            DEVICE,
            non_blocking=True
        )

        mask = batch[
            "mask"
        ].to(
            DEVICE,
            non_blocking=True
        )

        changeflag = batch[
            "changeflag"
        ].to(
            DEVICE,
            non_blocking=True
        )

        captions = batch[
            "caption"
        ].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            pred_mask, pred_flag, pred_caption = (
                model(
                    image_a,
                    image_b,
                    captions
                )
            )

            loss_seg = segmentation_loss(
                pred_mask,
                mask
            )

            loss_cls = bce_cls(
                pred_flag.squeeze(1),
                changeflag
            )

            loss_cap = caption_loss_fn(
                pred_caption.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                captions[:, 1:].reshape(
                    -1
                )
            )

            loss = (
                SEG_WEIGHT * loss_seg
                +
                CLS_WEIGHT * loss_cls
                +
                CAPTION_WEIGHT * loss_cap
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        total_loss += loss.item()
        total_seg += loss_seg.item()
        total_cls += loss_cls.item()
        total_cap += loss_cap.item()

        progress.set_postfix(
            loss=f"{loss.item():.3f}"
        )

    n = len(loader)

    return {
        "loss": total_loss / n,
        "seg": total_seg / n,
        "cls": total_cls / n,
        "cap": total_cap / n
    }


# ============================================================
# 19. VALIDATION
# ============================================================

@torch.no_grad()
def validate(
    model,
    loader
):

    model.eval()

    total_loss = 0.0
    total_seg = 0.0
    total_cls = 0.0
    total_cap = 0.0

    all_preds = []
    all_targets = []

    for batch in tqdm(
        loader,
        desc="Validation"
    ):

        image_a = batch[
            "image_a"
        ].to(
            DEVICE,
            non_blocking=True
        )

        image_b = batch[
            "image_b"
        ].to(
            DEVICE,
            non_blocking=True
        )

        mask = batch[
            "mask"
        ].to(
            DEVICE,
            non_blocking=True
        )

        changeflag = batch[
            "changeflag"
        ].to(
            DEVICE,
            non_blocking=True
        )

        captions = batch[
            "caption"
        ].to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            pred_mask, pred_flag, pred_caption = (
                model(
                    image_a,
                    image_b,
                    captions
                )
            )

            loss_seg = segmentation_loss(
                pred_mask,
                mask
            )

            loss_cls = bce_cls(
                pred_flag.squeeze(1),
                changeflag
            )

            loss_cap = caption_loss_fn(
                pred_caption.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                captions[:, 1:].reshape(
                    -1
                )
            )

            loss = (
                SEG_WEIGHT * loss_seg
                +
                CLS_WEIGHT * loss_cls
                +
                CAPTION_WEIGHT * loss_cap
            )

        total_loss += loss.item()
        total_seg += loss_seg.item()
        total_cls += loss_cls.item()
        total_cap += loss_cap.item()

        probs = torch.sigmoid(
            pred_flag.squeeze(1)
        )

        preds = (
            probs > 0.5
        ).long()

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_targets.extend(
            changeflag.cpu().numpy()
        )

    n = len(loader)

    accuracy = np.mean(
        np.array(all_preds)
        ==
        np.array(all_targets)
    )

    return {
        "loss": total_loss / n,
        "seg": total_seg / n,
        "cls": total_cls / n,
        "cap": total_cap / n,
        "accuracy": accuracy
    }


# ============================================================
# 20. TRAINING
# ============================================================

print("\n")
print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)

best_val_loss = float("inf")

history = []

for epoch in range(
    1,
    EPOCHS + 1
):

    print(
        f"\nEpoch {epoch}/{EPOCHS}"
    )

    train_metrics = train_one_epoch(
        model,
        train_loader
    )

    val_metrics = validate(
        model,
        val_loader
    )

    scheduler.step(
        val_metrics["loss"]
    )

    current_lr = optimizer.param_groups[0][
        "lr"
    ]

    print(
        "\nTrain:"
        f" {train_metrics['loss']:.4f}"
        f" | Seg {train_metrics['seg']:.4f}"
        f" | Cls {train_metrics['cls']:.4f}"
        f" | Cap {train_metrics['cap']:.4f}"
    )

    print(
        "Val:"
        f" {val_metrics['loss']:.4f}"
        f" | Seg {val_metrics['seg']:.4f}"
        f" | Cls {val_metrics['cls']:.4f}"
        f" | Cap {val_metrics['cap']:.4f}"
        f" | Acc {val_metrics['accuracy']:.4f}"
    )

    print(
        "LR:",
        f"{current_lr:.2e}"
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_seg": train_metrics["seg"],
        "train_cls": train_metrics["cls"],
        "train_cap": train_metrics["cap"],
        "val_loss": val_metrics["loss"],
        "val_seg": val_metrics["seg"],
        "val_cls": val_metrics["cls"],
        "val_cap": val_metrics["cap"],
        "val_accuracy": val_metrics["accuracy"],
        "lr": current_lr
    })

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": best_val_loss,
                "word2idx": word2idx,
                "idx2word": idx2word
            },
            "best_siamese_unet_caption_model.pth"
        )

        print(
            "✓ Best model saved"
        )


# ============================================================
# 21. SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    "training_history.csv",
    index=False
)

print("\n")
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Saved:"
)

print(
    "  best_siamese_unet_caption_model.pth"
)

print(
    "  training_history.csv"
)

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU

Dataset:
Total: 7264
split
train    6345
val       919
Name: count, dtype: int64

Change distribution:
split  changeflag
train  0             2938
       1             3407
val    0              452
       1              467
dtype: int64

Vocabulary size: 899

Train: 6345
Val: 919
Train batches: 1587
Val batches: 230

BATCH VERIFICATION
Image A: torch.Size([4, 3, 256, 256])
Image B: torch.Size([4, 3, 256, 256])
Mask: torch.Size([4, 1, 256, 256])
Caption: torch.Size([4, 40])
Change flag: torch.Size([4])

Trainable parameters: 2.82 M

MODEL VERIFICATION
Mask: torch.Size([2, 1, 256, 256])
Classification: torch.Size([2, 1])
Caption: torch.Size([2, 39, 899])

MODEL VERIFICATION: PASS


STARTING TRAINING

Epoch 1/100


Training:   0%|          | 0/1587 [00:00<?, ?it/s]

Validation:   0%|          | 0/230 [00:00<?, ?it/s]


Train: 3.6723 | Seg 0.6713 | Cls 0.5605 | Cap 2.7208
Val: 2.7124 | Seg 0.6107 | Cls 0.8069 | Cap 1.6983 | Acc 0.6953
LR: 1.00e-04
✓ Best model saved

Epoch 2/100


Training:   0%|          | 0/1587 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/230 [00:00<?, ?it/s]


Train: 1.5895 | Seg 0.5688 | Cls 0.1236 | Cap 0.9590
Val: 1.7907 | Seg 0.5680 | Cls 0.4839 | Cap 0.9808 | Acc 0.8226
LR: 1.22e-08

Epoch 100/100


Training:   0%|          | 0/1587 [00:00<?, ?it/s]

Validation:   0%|          | 0/230 [00:00<?, ?it/s]


Train: 1.5938 | Seg 0.5689 | Cls 0.1270 | Cap 0.9614
Val: 1.8220 | Seg 0.5682 | Cls 0.5411 | Cap 0.9832 | Acc 0.7987
LR: 1.22e-08


TRAINING COMPLETE
Best validation loss: 1.7156884965689285
Saved:
  best_siamese_unet_caption_model.pth
  training_history.csv
